# Multi-Stage GNN Code Security Pipeline

This notebook consolidates the entire pipeline for processing code security data using Graph Neural Networks. It includes data gathering, preprocessing, and training phases.

## Setup and Dependencies

In [ ]:
import os
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
import zipfile

print("Libraries imported successfully!")

## 1. Data Gathering

In [ ]:
def load_raw_data(directory_path):
    """
    Loads raw data from individual files in a directory into a pandas DataFrame.
    Each file is expected to contain a label and content separated by markers.
    """
    data_list = []
    print(f"Loading data from: {directory_path}")
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path):
            with open(file_path, 'r') as f:
                content = f.read()
                lines = content.splitlines()
                
                label = None
                if len(lines) > 1 and "-----label-----" in lines[0]:
                    try:
                        label = int(lines[1])
                    except ValueError:
                        label = None # Handle cases where label is not an integer

                code_start_index = -1
                for i, line in enumerate(lines):
                    if "-----code-----" in line:
                        code_start_index = i
                        break
                
                code_content = "\n".join(lines[code_start_index + 1:]) if code_start_index != -1 else content

                data_list.append({'filename': filename, 'label': label, 'contents': code_content})
    
    df = pd.DataFrame(data_list)
    print(f"Loaded {len(df)} entries.")
    return df

# Example usage of data gathering
local_dataset_path = 'dataset/CWE-77/5result/'

# Load data
if os.path.exists(local_dataset_path):
    df_raw = load_raw_data(local_dataset_path)
    print("Raw Data Head:")
    print(df_raw.head())
    print("\nRaw Data Info:")
    df_raw.info()
else:
    print(f"Local dataset not found at: {local_dataset_path}. Please ensure the dataset is in the correct location.")

## 2. Preprocessing

In [ ]:
def parse_graph_representation(contents):
    """
    Parses the raw content string to extract edges and raw node features.
    Expected delimiters: "-----children-----", "-----attribute-----", "-----ast_node-----", "-----joern-----".
    """
    edges = []
    node_features_raw = []
    current_section = None

    if contents:
        lines = contents.splitlines()
        for line in lines:
            line = line.strip()
            if line == "-----children-----":
                current_section = "children"
                continue
            elif line == "-----nextToken-----":
                current_section = "nextToken"
                continue
            elif line == "-----computeFrom-----":
                current_section = "computeFrom"
                continue
            elif line == "-----guardedBy-----":
                current_section = "guardedBy"
                continue
            elif line == "-----guardedByNegation-----":
                current_section = "guardedByNegation"
                continue
            elif line == "-----lastLexicalUse-----":
                current_section = "lastLexicalUse"
                continue
            elif line == "-----jump-----":
                current_section = "jump"
                continue
            elif line == "-----attribute-----":
                current_section = "attribute"
                continue
            elif line == "-----ast_node-----":
                current_section = "ast_node"
                continue
            elif line == "-----joern-----":
                current_section = "joern"
                continue
            elif line.startswith("-----"): # Handle other potential delimiters
                current_section = None
                continue

            if current_section == "children":
                match = re.match(r'(\d+),(\d+)', line)
                if match:
                    edges.append((int(match.group(1)), int(match.group(2))))
            elif current_section in ["attribute", "ast_node", "joern"]:
                if line: # Avoid adding empty lines
                    node_features_raw.append(line)
    
    return edges, node_features_raw


def create_graph_data_list(df):
    """
    Converts a DataFrame with 'edges', 'node_features', and 'label' columns
    into a list of PyTorch Geometric Data objects.
    """
    graph_data_list = []
    print("Creating graph data objects...")
    for index, row in df.iterrows():
        edges = row['edges']
        node_features_raw = row['node_features']
        label = row['label']

        if not edges:
            continue # Skip if there are no edges

        # Create a mapping from original node indices to continuous indices
        unique_nodes = sorted(list(set([node for edge in edges for node in edge])))
        node_mapping = {old_index: new_index for new_index, old_index in enumerate(unique_nodes)}

        # Re-index edges
        reindexed_edges = [(node_mapping[u], node_mapping[v]) for u, v in edges]
        edge_index = torch.tensor(reindexed_edges, dtype=torch.long).t().contiguous()

        # Process node features (basic approach: use raw strings as features for now)
        # This part will need refinement for a real GNN
        num_nodes = len(unique_nodes)
        node_features = torch.ones((num_nodes, 1), dtype=torch.float) # Placeholder

        # Create PyTorch Geometric Data object
        data = Data(edge_index=edge_index, x=node_features, y=torch.tensor([label], dtype=torch.float))
        graph_data_list.append(data)
    
    print(f"Created {len(graph_data_list)} graph data objects.")
    return graph_data_list

# Process the data to create graph structures
if 'df_raw' in locals():
    df_raw[['edges', 'node_features']] = df_raw['contents'].apply(lambda x: pd.Series(parse_graph_representation(x)))
    graph_data_list = create_graph_data_list(df_raw)
    print(f"Graph data list created with {len(graph_data_list)} entries.")

## 3. Model Definition

In [ ]:
class GCN(nn.Module):
    """
    A Graph Convolutional Network (GCN) model for graph classification.
    """
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(-1, hidden_channels) # -1 for inferred input features
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, 1) # Binary classification output

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # 1. Obtain node embeddings
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)

        # 2. Readout layer
        x = global_mean_pool(x, batch)  # Aggregate node features to graph features

        # 3. Apply a final classifier
        x = nn.functional.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return torch.sigmoid(x) # Sigmoid for binary classification

# Initialize the model if we have graph data
if 'graph_data_list' in locals() and len(graph_data_list) > 0:
    model = GCN(hidden_channels=64)
    print("Model initialized successfully!")
    print(model)

## 4. Training

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs=30, learning_rate=0.001):
    """
    Trains and evaluates the GNN model.
    """
    criterion = nn.BCEWithLogitsLoss() # Use BCEWithLogitsLoss for sigmoid output
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print("Starting model training...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for data in train_loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y.unsqueeze(1)) # Ensure target has a trailing dimension of 1
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        # Testing phase
        model.eval()
        total_test_loss = 0
        with torch.no_grad():
            for data in test_loader:
                out = model(data)
                loss = criterion(out, data.y.unsqueeze(1))
                total_test_loss += loss.item()

        avg_test_loss = total_test_loss / len(test_loader)
        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}')
    print("Training complete.")


def evaluate_model(model, test_loader):
    """
    Evaluates the trained model on the test dataset and prints accuracy.
    """
    model.eval()
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for data in test_loader:
            out = model(data)
            predictions = (out > 0.5).float() # Apply threshold for binary prediction
            correct_predictions += (predictions == data.y.unsqueeze(1)).sum().item()
            total_samples += data.y.size(0)

    accuracy = correct_predictions / total_samples
    print(f'Test Accuracy: {accuracy:.4f}')
    return accuracy


def make_predictions(model, test_dataset, num_samples=5):
    """
    Makes predictions on a few samples from the test dataset.
    """
    selected_samples = test_dataset[:num_samples]
    model.eval()
    print(f"\nPredictions on selected {num_samples} test samples:")
    with torch.no_grad():
        for i, data in enumerate(selected_samples):
            out = model(data)
            prediction = (out > 0.5).float().item() # Apply threshold and get scalar value
            actual_label = data.y.item() # Get scalar value of the actual label
            print(f"Sample {i+1}: Predicted Label = {int(prediction)}, Actual Label = {int(actual_label)}")


# Split dataset if we have graph data
if 'graph_data_list' in locals() and len(graph_data_list) > 0:
    train_dataset, test_dataset = train_test_split(graph_data_list, test_size=0.2, random_state=42)
    print(f"Training dataset size: {len(train_dataset)}")
    print(f"Testing dataset size: {len(test_dataset)}")

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Initialize model
    model = GCN(hidden_channels=64)

    # Train the model
    train_model(model, train_loader, test_loader, num_epochs=30)

    # Evaluate the model
    accuracy = evaluate_model(model, test_loader)

    # Make predictions on test samples
    make_predictions(model, test_dataset)

## Summary

This notebook consolidates the entire multi-stage GNN code security pipeline:
1. Data Gathering: Loading raw data from files
2. Preprocessing: Converting code representations to graph structures
3. Training: Training a Graph Neural Network to detect security vulnerabilities
.